# Visualización genérica de datos de Google Earth Engine

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/scantoni-cmd/example_unal/blob/main/notebooks/gee_data_visualization.ipynb)

Este notebook consulta, procesa y visualiza datasets del catálogo de **Google Earth Engine (GEE)** con `earthengine-api` y `geemap`.
Solo debes cambiar unos pocos parámetros (proyecto, área de estudio, período) y ejecutar todo.

**Contenido**

| # | Ejemplo | Dataset |
|---|---------|---------|
| 1 | Cobertura del suelo | `ESA/WorldCover/v200` |
| 2 | Elevación, pendiente y sombreado | `USGS/SRTMGL1_003` |
| 3 | Precipitación acumulada y media | `UCSB-CHG/CHIRPS/DAILY` |
| 4 | Temperatura del aire | `ECMWF/ERA5_LAND/DAILY_AGGR` |
| 5 | Mapa combinado y *split map* | los cuatro anteriores |

**Antes de empezar:** necesitas un proyecto de Google Cloud registrado para Earth Engine (ver `GOOGLE_CLOUD_SETUP.md`).

## 1. Instalación y entorno

In [ ]:
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "earthengine-api", "geemap"]
    )
    print("Dependencias instaladas en Colab.")
else:
    print("Ejecución local: si falta algo, usa `pip install -r requirements.txt`.")

In [ ]:
import json
import os
import urllib.request
from dataclasses import dataclass
from typing import Any, Optional

import ee
import geemap
import pandas as pd

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
print("earthengine-api", ee.__version__, "| geemap", geemap.__version__)

## 2. Configuración global

Cambia **solo esta celda** para adaptar el notebook.

- `PROJECT_ID`: tu proyecto de Google Cloud con Earth Engine. También se lee de la variable de entorno
  `EE_PROJECT_ID` o de un *Secret* de Colab con ese nombre (tienen prioridad).
- `AOI_METHOD`: `"bbox"`, `"polygon"`, `"geojson"` o `"admin"`.
- `end_date` es **exclusiva** (el día final no se incluye).

In [ ]:
# ---- Proyecto ------------------------------------------------------------
PROJECT_ID = "tu-proyecto-gee"          # <-- CAMBIA ESTO

# ---- Área de estudio (AOI) -----------------------------------------------
AOI_METHOD = "bbox"                      # "bbox" | "polygon" | "geojson" | "admin"

# (lon_min, lat_min, lon_max, lat_max). Por defecto: Cali y alrededores (~45 x 40 km)
AOI_BBOX = (-76.80, 3.25, -76.39, 3.61)

# Lista de vértices (lon, lat)
AOI_POLYGON = [(-76.80, 3.25), (-76.39, 3.25), (-76.39, 3.61), (-76.80, 3.61)]

# Ruta local o URL raw de un .geojson. Si no existe y se llama aoi_example.geojson, se genera solo.
AOI_GEOJSON = "data/aoi_example.geojson"

# División administrativa FAO GAUL 2025
ADMIN_COUNTRY = "Colombia"
ADMIN_REGION = "Valle del Cauca"         # None para usar el país completo
ADMIN_LEVEL = 1                          # 0 = país, 1 = departamento/estado, 2 = municipio

# ---- Período para series temporales (CHIRPS, ERA5-Land) --------------------
PERIOD_START = "2024-03-01"
PERIOD_END = "2024-06-01"                # exclusiva

# ---- Exportación opcional a Google Drive ------------------------------------
EXPORT_TO_DRIVE = False
EXPORT_FOLDER = "GEE_exports"

## 3. Funciones de autenticación, errores y validación

In [ ]:
def _resolve_project(default):
    """Prioridad: variable de entorno > Secret de Colab > valor del notebook."""
    env = os.environ.get("EE_PROJECT_ID")
    if env:
        return env
    if IN_COLAB:
        try:
            from google.colab import userdata
            val = userdata.get("EE_PROJECT_ID")
            if val:
                return val
        except Exception:
            pass
    return default


def translate_ee_error(exc):
    """Convierte errores típicos de Earth Engine en mensajes accionables en español."""
    msg = str(exc)
    low = msg.lower()
    rules = [
        (("not signed up", "not registered", "project is not registered"),
         "El proyecto no está registrado para Earth Engine. Regístralo en https://code.earthengine.google.com/register"),
        (("api has not been used", "service_disabled", "has not been enabled"),
         "La Earth Engine API no está habilitada en el proyecto. Habilítala en Google Cloud Console."),
        (("permission", "403", "caller does not have", "forbidden"),
         "Faltan permisos en el proyecto. Necesitas roles/serviceusage.serviceUsageConsumer y roles/earthengine.viewer."),
        (("quota", "too many concurrent", "429", "rate limit"),
         "Se superó la cuota de Earth Engine. Espera unos minutos o reduce el AOI."),
        (("user memory limit", "memory limit exceeded"),
         "La consulta usa demasiada memoria. Reduce el AOI, aumenta `scale` o acorta el período."),
        (("timed out", "timeout"),
         "La consulta tardó demasiado. Reduce el AOI o el período, o aumenta `scale`."),
        (("band pattern", "has no band", "no bands", "was not found"),
         "La banda indicada no existe en el dataset. Revisa el nombre en el catálogo de GEE."),
        (("asset", "does not exist", "not found"),
         "El dataset (asset) no existe o no tienes acceso. Revisa el ID en el catálogo de GEE."),
    ]
    for keys, text in rules:
        if any(k in low for k in keys):
            return f"{text}\n\nDetalle original: {msg}"
    return msg


def initialize_ee(project=None):
    """Inicializa Earth Engine; si no hay credenciales, lanza la autenticación interactiva."""
    project = _resolve_project(project or PROJECT_ID)
    if not project or project == "tu-proyecto-gee":
        raise ValueError(
            "Define PROJECT_ID en la celda de configuración (o EE_PROJECT_ID como variable de entorno / Secret de Colab)."
        )
    try:
        ee.Initialize(project=project)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=project)
    return project


def verify_connection():
    """Prueba mínima contra el servidor de EE."""
    try:
        ok = ee.Number(1).add(1).getInfo() == 2
    except Exception as exc:
        raise RuntimeError(translate_ee_error(exc)) from exc
    print("Conexión con Earth Engine OK." if ok else "Respuesta inesperada de Earth Engine.")
    return ok


def validate_bbox(bbox):
    """bbox = (lon_min, lat_min, lon_max, lat_max) en grados."""
    try:
        w, s, e, n = [float(v) for v in bbox]
    except (TypeError, ValueError):
        raise ValueError("bbox debe ser (lon_min, lat_min, lon_max, lat_max) con 4 números.")
    if not (-180 <= w < e <= 180):
        raise ValueError(f"Longitudes inválidas: {w}, {e}. Deben cumplir -180 <= lon_min < lon_max <= 180.")
    if not (-90 <= s < n <= 90):
        raise ValueError(f"Latitudes inválidas: {s}, {n}. Deben cumplir -90 <= lat_min < lat_max <= 90.")
    return w, s, e, n


def validate_polygon(vertices):
    """Lista de (lon, lat) con al menos 3 vértices distintos."""
    try:
        pts = [(float(x), float(y)) for x, y in vertices]
    except (TypeError, ValueError):
        raise ValueError("El polígono debe ser una lista de pares (lon, lat).")
    if len(pts) > 1 and pts[0] == pts[-1]:
        pts = pts[:-1]
    if len(set(pts)) < 3:
        raise ValueError("El polígono necesita al menos 3 vértices distintos.")
    for x, y in pts:
        if not (-180 <= x <= 180 and -90 <= y <= 90):
            raise ValueError(f"Vértice fuera de rango: ({x}, {y}). Orden esperado: (lon, lat).")
    return pts


def check_dates(start_date, end_date):
    """Valida el rango de fechas; devuelve (inicio, fin, días)."""
    if start_date is None and end_date is None:
        return None, None, None
    if start_date is None or end_date is None:
        raise ValueError("Debes indicar start_date y end_date juntas.")
    try:
        s, e = pd.Timestamp(start_date), pd.Timestamp(end_date)
    except Exception:
        raise ValueError("Formato de fecha inválido. Usa 'YYYY-MM-DD'.")
    if s >= e:
        raise ValueError("start_date debe ser anterior a end_date (end_date es exclusiva).")
    return s.strftime("%Y-%m-%d"), e.strftime("%Y-%m-%d"), (e - s).days

## 4. Área de estudio (AOI)

In [ ]:
EXAMPLE_GEOJSON = {
    "type": "FeatureCollection",
    "features": [{
        "type": "Feature",
        "properties": {"name": "AOI ejemplo - Cali, Valle del Cauca"},
        "geometry": {
            "type": "Polygon",
            "coordinates": [[[-76.80, 3.25], [-76.39, 3.25], [-76.39, 3.61], [-76.80, 3.61], [-76.80, 3.25]]],
        },
    }],
}

GAUL_ASSET = "FAO/GAUL/2025/level{level}"


def load_geojson(src):
    """Lee un GeoJSON (dict, ruta local o URL) y devuelve un ee.Geometry."""
    if isinstance(src, dict):
        data = src
    else:
        src = str(src)
        if src.startswith(("http://", "https://")):
            with urllib.request.urlopen(src, timeout=30) as resp:
                data = json.loads(resp.read().decode("utf-8"))
        elif os.path.exists(src):
            with open(src, encoding="utf-8") as fh:
                data = json.load(fh)
        elif os.path.basename(src) == "aoi_example.geojson":
            data = EXAMPLE_GEOJSON
            try:
                os.makedirs(os.path.dirname(src) or ".", exist_ok=True)
                with open(src, "w", encoding="utf-8") as fh:
                    json.dump(EXAMPLE_GEOJSON, fh, indent=2)
                print(f"No existía {src}: se generó el GeoJSON de ejemplo.")
            except OSError:
                print("Se usa el GeoJSON de ejemplo embebido (no se pudo escribir en disco).")
        else:
            raise FileNotFoundError(f"No se encontró el GeoJSON: {src}")

    kind = data.get("type")
    if kind == "FeatureCollection":
        feats = [ee.Feature(f) for f in data["features"]]
        if not feats:
            raise ValueError("El GeoJSON no contiene features.")
        return ee.FeatureCollection(feats).geometry()
    if kind == "Feature":
        return ee.Feature(data).geometry()
    return ee.Geometry(data)


def aoi_from_admin(country, region=None, level=1):
    """Geometría de una división administrativa de FAO GAUL 2025."""
    if region is not None and level < 1:
        raise ValueError("Para filtrar por región usa ADMIN_LEVEL >= 1.")
    fc = ee.FeatureCollection(GAUL_ASSET.format(level=level))
    names = ee.Feature(fc.first()).propertyNames().getInfo()
    lower = {n.lower(): n for n in names}

    def pick(*candidates):
        for c in candidates:
            if c.lower() in lower:
                return lower[c.lower()]
        raise KeyError(f"No encontré ninguno de {candidates} entre los campos de GAUL: {names}")

    fc = fc.filter(ee.Filter.eq(pick("ADM0_NAME", "gaul0_name"), country))
    if region:
        fc = fc.filter(ee.Filter.eq(pick(f"ADM{level}_NAME", f"gaul{level}_name"), region))
    if fc.size().getInfo() == 0:
        raise ValueError(f"No se encontró '{region or country}' en GAUL (nivel {level}). Revisa la ortografía.")
    return fc.geometry()


def to_geometry(aoi):
    """Acepta ee.Geometry, bbox (4 números), lista de vértices, dict GeoJSON o ruta/URL a un GeoJSON."""
    if isinstance(aoi, ee.Geometry):
        return aoi
    if isinstance(aoi, (ee.Feature, ee.FeatureCollection)):
        return aoi.geometry()
    if isinstance(aoi, dict) or isinstance(aoi, str):
        return load_geojson(aoi)
    if isinstance(aoi, (list, tuple)):
        if len(aoi) == 4 and all(isinstance(v, (int, float)) for v in aoi):
            w, s, e, n = validate_bbox(aoi)
            return ee.Geometry.Rectangle([w, s, e, n], geodesic=False)
        return ee.Geometry.Polygon([validate_polygon(aoi)], geodesic=False)
    raise TypeError(f"Tipo de AOI no soportado: {type(aoi)}")


def resolve_aoi(method=None):
    """Construye el AOI según la configuración global."""
    method = (method or AOI_METHOD).lower()
    if method == "bbox":
        return to_geometry(validate_bbox(AOI_BBOX))
    if method == "polygon":
        return to_geometry(validate_polygon(AOI_POLYGON))
    if method == "geojson":
        return load_geojson(AOI_GEOJSON)
    if method == "admin":
        return aoi_from_admin(ADMIN_COUNTRY, ADMIN_REGION, ADMIN_LEVEL)
    raise ValueError("AOI_METHOD debe ser 'bbox', 'polygon', 'geojson' o 'admin'.")

## 5. Función genérica de visualización

In [ ]:
REDUCERS = {
    "mean": lambda c: c.mean(),
    "sum": lambda c: c.sum(),
    "median": lambda c: c.median(),
    "min": lambda c: c.min(),
    "max": lambda c: c.max(),
    "mode": lambda c: c.mode(),
    "mosaic": lambda c: c.mosaic(),
    "first": lambda c: c.first(),
}
NO_DATES_OK = {"mosaic", "first"}
DEFAULT_PALETTE = ["440154", "3b528b", "21908d", "5dc863", "fde725"]


def _normalize_dataset_type(dtype):
    key = str(dtype).strip().lower().replace("_", "").replace(" ", "")
    if key in ("auto", ""):
        return "auto"
    if key == "image":
        return "Image"
    if key in ("imagecollection", "collection", "ic"):
        return "ImageCollection"
    raise ValueError("dataset_type debe ser 'auto', 'Image' o 'ImageCollection'.")


def _asset_type(dataset_id):
    info = ee.data.getAsset(dataset_id)
    return {"IMAGE": "Image", "IMAGE_COLLECTION": "ImageCollection"}.get(info.get("type"), info.get("type"))


def compute_statistics(image, band, geometry, scale):
    """Mínimo, máximo, media, desviación estándar y número de píxeles dentro del AOI."""
    reducer = (
        ee.Reducer.min()
        .combine(ee.Reducer.max(), sharedInputs=True)
        .combine(ee.Reducer.mean(), sharedInputs=True)
        .combine(ee.Reducer.stdDev(), sharedInputs=True)
        .combine(ee.Reducer.count(), sharedInputs=True)
    )
    r = image.select([band]).reduceRegion(
        reducer=reducer, geometry=geometry, scale=scale, bestEffort=True, maxPixels=1e9, tileScale=4
    ).getInfo()
    return {
        "mínimo": r.get(f"{band}_min"),
        "máximo": r.get(f"{band}_max"),
        "media": r.get(f"{band}_mean"),
        "desv. estándar": r.get(f"{band}_stdDev"),
        "n píxeles": r.get(f"{band}_count"),
    }


def _auto_range(image, band, geometry, scale, low=2, high=98):
    r = image.select([band]).reduceRegion(
        reducer=ee.Reducer.percentile([low, high]),
        geometry=geometry, scale=scale, bestEffort=True, maxPixels=1e9, tileScale=4,
    ).getInfo()
    lo, hi = r.get(f"{band}_p{low}"), r.get(f"{band}_p{high}")
    if lo is None or hi is None:
        raise ValueError("No hay píxeles con datos dentro del AOI para ese dataset/período.")
    if lo == hi:
        hi = lo + 1
    return lo, hi


def new_map(aoi=None, height="650px", draw_aoi=True):
    """Crea un geemap.Map centrado en el AOI y con su contorno."""
    m = geemap.Map(height=height)
    if aoi is not None:
        geom = to_geometry(aoi)
        m.centerObject(geom)
        if draw_aoi:
            outline = ee.FeatureCollection([ee.Feature(geom)]).style(
                color="ff0000", fillColor="00000000", width=2
            )
            m.add_layer(outline, {}, "AOI")
    return m


@dataclass
class VisResult:
    dataset_id: str
    band: str
    image: Any                 # imagen con valores reales (para estadísticas/exportación)
    display_image: Any         # imagen que se dibuja (puede estar reclasificada)
    vis_params: dict
    aoi: Any
    layer_name: str
    scale: float
    statistics: Optional[dict] = None
    map: Any = None
    legend: Optional[dict] = None
    colorbar_label: Optional[str] = None

    def add_to(self, m, show=True, colorbar=False, colorbar_position="bottomright"):
        """Agrega esta capa (y opcionalmente leyenda/barra de color) a otro mapa."""
        m.add_layer(self.display_image, self.vis_params, self.layer_name, show)
        if show and self.legend:
            try:
                m.add_legend(title=self.colorbar_label or self.layer_name, legend_dict=self.legend)
            except Exception as exc:
                print(f"(No se pudo añadir la leyenda: {exc})")
        elif show and colorbar and self.vis_params.get("palette"):
            try:
                m.add_colorbar(
                    vis_params=self.vis_params, label=self.colorbar_label or self.layer_name,
                    position=colorbar_position,
                )
            except Exception as exc:
                print(f"(No se pudo añadir la barra de color: {exc})")
        return m


def visualize_gee_dataset(
    dataset_id,
    band,
    aoi,
    dataset_type="auto",
    start_date=None,
    end_date=None,
    reducer="mean",
    vis_params=None,
    scale=None,
    scale_factor=None,
    offset=None,
    clip=True,
    layer_name=None,
    colorbar_label=None,
    colorbar_position="bottomright",
    class_values=None,
    class_names=None,
    class_palette=None,
    compute_stats=False,
    m=None,
    add_to_map=True,
    show_colorbar=True,
):
    """Visualiza cualquier Image/ImageCollection de GEE sobre un AOI.

    Valores finales = (valor * scale_factor) + offset. Para capas categóricas pasa
    class_values, class_names y class_palette (misma longitud).
    """
    if reducer not in REDUCERS:
        raise ValueError(f"reducer debe ser uno de {sorted(REDUCERS)}")
    start_date, end_date, _ = check_dates(start_date, end_date)

    try:
        geom = to_geometry(aoi)
        dtype = _normalize_dataset_type(dataset_type)
        if dtype == "auto":
            dtype = _asset_type(dataset_id)

        if dtype == "Image":
            src = ee.Image(dataset_id).select([band])
            img = src
        elif dtype == "ImageCollection":
            if start_date is None and reducer not in NO_DATES_OK:
                raise ValueError(
                    f"Para agregar con '{reducer}' una ImageCollection debes indicar start_date y end_date."
                )
            col = ee.ImageCollection(dataset_id).filterBounds(geom)
            if start_date:
                col = col.filterDate(start_date, end_date)
            col = col.select([band])
            n_images = col.size().getInfo()
            if n_images == 0:
                raise ValueError(
                    "No hay imágenes para ese AOI y período. Algunos productos (CHIRPS, ERA5-Land) se "
                    "publican con rezago: prueba fechas más antiguas."
                )
            src = col.first()
            img = REDUCERS[reducer](col)
        else:
            raise ValueError(f"Tipo de asset no soportado: {dtype}")

        img = ee.Image(img).select([band])
        if scale_factor is not None:
            img = img.multiply(scale_factor)
        if offset is not None:
            img = img.add(offset)
        if clip:
            img = img.clip(geom)

        if scale is None:
            scale = ee.Image(src).select([band]).projection().nominalScale().getInfo()

        legend = None
        vis = dict(vis_params or {})
        if class_values is not None:
            if not (class_names and class_palette and len(class_values) == len(class_names) == len(class_palette)):
                raise ValueError("class_values, class_names y class_palette deben tener la misma longitud.")
            display = img.remap(list(class_values), list(range(len(class_values)))).rename(band)
            vis = {"min": 0, "max": len(class_values) - 1, "palette": list(class_palette)}
            legend = {f"{v} {n}": c for v, n, c in zip(class_values, class_names, class_palette)}
        else:
            display = img
            if "min" not in vis or "max" not in vis:
                lo, hi = _auto_range(img, band, geom, scale)
                vis.setdefault("min", lo)
                vis.setdefault("max", hi)
            vis.setdefault("palette", DEFAULT_PALETTE)

        stats = compute_statistics(img, band, geom, scale) if compute_stats else None
    except ee.EEException as exc:
        raise RuntimeError(translate_ee_error(exc)) from exc

    name = layer_name or f"{band} ({reducer})"
    res = VisResult(
        dataset_id=dataset_id, band=band, image=img, display_image=display, vis_params=vis,
        aoi=geom, layer_name=name, scale=scale, statistics=stats, legend=legend,
        colorbar_label=colorbar_label,
    )
    if add_to_map:
        if m is None:
            m = new_map(geom)
        res.add_to(m, show=True, colorbar=show_colorbar, colorbar_position=colorbar_position)
        res.map = m
    return res

## 6. Registro de datasets

Cada ficha define cómo visualizar un dataset. Para agregar uno nuevo, añade una entrada con la misma estructura
(solo necesitas `dataset_id`, `band` y los parámetros de visualización).

In [ ]:
WORLDCOVER_CLASSES = {
    10: ("Árboles", "006400"),
    20: ("Matorral", "ffbb22"),
    30: ("Pastizal", "ffff4c"),
    40: ("Cultivos", "f096ff"),
    50: ("Construido", "fa0000"),
    60: ("Suelo desnudo / vegetación escasa", "b4b4b4"),
    70: ("Nieve y hielo", "f0f0f0"),
    80: ("Agua permanente", "0064c8"),
    90: ("Humedal herbáceo", "0096a0"),
    95: ("Manglar", "00cf75"),
    100: ("Musgo y líquenes", "fae6a0"),
}

DATASET_REGISTRY = {
    "landcover_worldcover": dict(
        dataset_id="ESA/WorldCover/v200", band="Map", dataset_type="ImageCollection", reducer="mosaic",
        scale=10, layer_name="Cobertura del suelo (ESA WorldCover 2021)", colorbar_label="Cobertura del suelo",
        class_values=list(WORLDCOVER_CLASSES),
        class_names=[v[0] for v in WORLDCOVER_CLASSES.values()],
        class_palette=[v[1] for v in WORLDCOVER_CLASSES.values()],
    ),
    "elevation_srtm": dict(
        dataset_id="USGS/SRTMGL1_003", band="elevation", dataset_type="Image",
        layer_name="Elevación SRTM (m)", colorbar_label="Elevación (m)",
        vis_params={"palette": ["006633", "e5ffcc", "f2cc8f", "b07040", "ffffff"]},
    ),
    "precip_chirps_sum": dict(
        dataset_id="UCSB-CHG/CHIRPS/DAILY", band="precipitation", dataset_type="ImageCollection", reducer="sum",
        layer_name="Precipitación acumulada (mm)", colorbar_label="Precipitación acumulada (mm)",
        vis_params={"palette": ["f7fbff", "6baed6", "08306b"]},
    ),
    "precip_chirps_mean": dict(
        dataset_id="UCSB-CHG/CHIRPS/DAILY", band="precipitation", dataset_type="ImageCollection", reducer="mean",
        layer_name="Precipitación media (mm/d)", colorbar_label="Precipitación media (mm/d)",
        vis_params={"palette": ["ffffe5", "74c476", "00441b"]},
    ),
    "temp_era5land": dict(
        dataset_id="ECMWF/ERA5_LAND/DAILY_AGGR", band="temperature_2m", dataset_type="ImageCollection",
        reducer="mean", offset=-273.15,    # K -> °C
        layer_name="Temperatura media del aire (°C)", colorbar_label="Temperatura media (°C)",
        vis_params={"palette": ["313695", "74add1", "ffffbf", "f46d43", "a50026"]},
    ),
}


def visualize_from_registry(key, aoi, **overrides):
    """Usa una ficha de DATASET_REGISTRY; cualquier parámetro puede sobrescribirse."""
    if key not in DATASET_REGISTRY:
        raise KeyError(f"'{key}' no está en el registro. Disponibles: {list(DATASET_REGISTRY)}")
    cfg = {**DATASET_REGISTRY[key], **overrides}
    return visualize_gee_dataset(aoi=aoi, **cfg)


def landcover_area_table(aoi, scale=10):
    """Área (ha) y porcentaje por clase de ESA WorldCover dentro del AOI."""
    geom = to_geometry(aoi)
    wc = ee.ImageCollection("ESA/WorldCover/v200").mosaic().select(["Map"])
    area_ha = ee.Image.pixelArea().divide(10000)
    try:
        out = area_ha.addBands(wc).reduceRegion(
            reducer=ee.Reducer.sum().group(groupField=1, groupName="clase"),
            geometry=geom, scale=scale, bestEffort=True, maxPixels=1e10, tileScale=4,
        ).getInfo()
    except ee.EEException as exc:
        raise RuntimeError(translate_ee_error(exc)) from exc
    df = pd.DataFrame(out["groups"]).rename(columns={"sum": "área (ha)"})
    df["clase"] = df["clase"].astype(int)
    df["nombre"] = df["clase"].map(lambda c: WORLDCOVER_CLASSES.get(c, ("Desconocida",))[0])
    df["%"] = 100 * df["área (ha)"] / df["área (ha)"].sum()
    return df[["clase", "nombre", "área (ha)", "%"]].sort_values("área (ha)", ascending=False).reset_index(drop=True)


def export_result_to_drive(res, description, folder="GEE_exports", scale=None, crs="EPSG:4326"):
    """Exporta una imagen (valores reales) a Google Drive como GeoTIFF."""
    task = ee.batch.Export.image.toDrive(
        image=res.image, description=description, folder=folder,
        region=res.aoi, scale=scale or res.scale, crs=crs, maxPixels=1e10, fileFormat="GeoTIFF",
    )
    task.start()
    print(f"Tarea '{description}' enviada. Revisa https://code.earthengine.google.com/tasks")
    return task


def split_compare(res_left, res_right, height="650px"):
    """Mapa con cortina: izquierda vs. derecha."""
    import ipyleaflet

    m = new_map(res_left.aoi, height=height, draw_aoi=False)
    m.add_layer(res_left.display_image, res_left.vis_params, res_left.layer_name)
    m.add_layer(res_right.display_image, res_right.vis_params, res_right.layer_name)
    left = m.find_layer(res_left.layer_name)
    right = m.find_layer(res_right.layer_name)
    m.add(ipyleaflet.SplitMapControl(left_layer=left, right_layer=right))
    return m

## 7. Conexión con Earth Engine y AOI

In [ ]:
PROJECT_ID = initialize_ee(PROJECT_ID)
verify_connection()
print("Proyecto:", PROJECT_ID)

In [ ]:
AOI = resolve_aoi()
print("AOI listo. Área aproximada (km²):", round(AOI.area(maxError=100).divide(1e6).getInfo(), 1))

## Ejemplo 1. Cobertura del suelo (ESA WorldCover)

Mapa categórico con leyenda oficial y tabla de área por clase.

In [ ]:
res_lc = visualize_from_registry("landcover_worldcover", AOI)
res_lc.map

In [ ]:
df_lc = landcover_area_table(AOI)
df_lc

## Ejemplo 2. Elevación (SRTM)

Estadísticas, pendiente (grados) y sombreado.

In [ ]:
res_dem = visualize_from_registry("elevation_srtm", AOI, compute_stats=True)

dem = ee.Image("USGS/SRTMGL1_003").select(["elevation"])
slope = ee.Terrain.slope(dem).clip(AOI)
hillshade = ee.Terrain.hillshade(dem).clip(AOI)

res_dem.map.add_layer(slope, {"min": 0, "max": 45, "palette": ["ffffcc", "fd8d3c", "800026"]}, "Pendiente (°)", False)
res_dem.map.add_layer(hillshade, {"min": 0, "max": 255}, "Sombreado", False)
res_dem.map.add_layer_control()

pd.DataFrame([res_dem.statistics], index=["Elevación (m)"])

In [ ]:
res_dem.map

In [ ]:
slope_stats = compute_statistics(slope, "slope", AOI, res_dem.scale)
pd.DataFrame([slope_stats], index=["Pendiente (°)"])

## Ejemplo 3. Precipitación (CHIRPS)

Acumulado (`sum`, mm) y promedio diario (`mean`, mm/d) del período `PERIOD_START`–`PERIOD_END`.

In [ ]:
res_p_sum = visualize_from_registry(
    "precip_chirps_sum", AOI, start_date=PERIOD_START, end_date=PERIOD_END, compute_stats=True
)
res_p_mean = visualize_from_registry(
    "precip_chirps_mean", AOI, start_date=PERIOD_START, end_date=PERIOD_END, compute_stats=True,
    m=res_p_sum.map, colorbar_position="bottomleft",
)
res_p_sum.map.add_layer_control()

pd.DataFrame(
    [res_p_sum.statistics, res_p_mean.statistics],
    index=["Acumulada (mm)", "Media (mm/d)"],
)

In [ ]:
res_p_sum.map

## Ejemplo 4. Temperatura (ERA5-Land)

Media del período, convertida de Kelvin a °C.

In [ ]:
res_t = visualize_from_registry(
    "temp_era5land", AOI, start_date=PERIOD_START, end_date=PERIOD_END, compute_stats=True
)
pd.DataFrame([res_t.statistics], index=["Temperatura (°C)"])

In [ ]:
res_t.map

## Ejemplo 5. Mapa combinado

Todas las capas en un solo mapa con control de capas e inspector, más un *split map* (cortina) para comparar.

In [ ]:
m_all = new_map(AOI)
for r in (res_lc, res_dem, res_p_sum, res_p_mean, res_t):
    r.add_to(m_all, show=(r is res_lc))
m_all.add_layer_control()

try:
    m_all.add_inspector()
except Exception:
    try:
        m_all.add("inspector")
    except Exception as exc:
        print(f"(Inspector no disponible en esta versión de geemap: {exc})")

m_all

In [ ]:
# Cortina: cobertura del suelo (izquierda) vs. elevación (derecha)
split_compare(res_lc, res_dem)

## Exportación opcional a Google Drive

In [ ]:
if EXPORT_TO_DRIVE:
    export_result_to_drive(res_dem, "elevacion_srtm_aoi", folder=EXPORT_FOLDER)
    export_result_to_drive(res_p_sum, "precipitacion_acumulada_aoi", folder=EXPORT_FOLDER)
    export_result_to_drive(res_t, "temperatura_media_aoi", folder=EXPORT_FOLDER)
else:
    print("Exportación desactivada. Pon EXPORT_TO_DRIVE = True en la configuración para activarla.")

## Próximos pasos

- Cambia `AOI_METHOD`/`AOI_BBOX` para usar otra zona.
- Agrega datasets al `DATASET_REGISTRY` y llámalos con `visualize_from_registry(...)`.
- Cualquier otro dataset: `visualize_gee_dataset(dataset_id=..., band=..., aoi=AOI, ...)`.

**Fuentes y licencias:** ESA WorldCover (CC-BY-4.0), NASA SRTM, CHIRPS v2.0, ERA5-Land (Copernicus), FAO GAUL 2025 (CC-BY-4.0).